# Building a Simple Agent with the Anthropic SDK

This notebook demonstrates the core concepts of building an agent:

1. A well-structured system prompt
2. Few-shot examples baked into the prompt
3. Evidence-based prompting (passing real data for the agent to reason about)
4. Tool use (giving the agent a tool it can call)
5. Proper input/output handling

This is a property analysis agent that can look up comparable sales and provide pricing recommendations based on evidence.

## Setup

Install the Anthropic SDK and initialize the client. The API key is loaded from an environment variable -- never hardcode keys.

In [ ]:
!pip install anthropic

In [ ]:
import os
import json
import anthropic

# Setup: Initialize the Anthropic client
# The API key is loaded from an environment variable -- never hardcode keys.
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

## Tool Definitions

Tools are functions the agent can call. You describe them with a JSON schema so the model knows what parameters to pass. The model doesn't execute the tool -- YOU do. The model just decides when to call it and with what args.

In [ ]:
tools = [
    {
        "name": "lookup_comparable_sales",
        "description": (
            "Look up recent comparable property sales in a given neighborhood. "
            "Returns a list of recent sales with address, price, sqft, bedrooms, "
            "bathrooms, and sale date. Use this tool when you need market data "
            "to support a pricing recommendation."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "neighborhood": {
                    "type": "string",
                    "description": "The neighborhood to search (e.g., 'Kailua', 'Manoa', 'Hawaii Kai')",
                },
                "bedrooms": {
                    "type": "integer",
                    "description": "Number of bedrooms to filter by",
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of comparable sales to return (default 5)",
                },
            },
            "required": ["neighborhood", "bedrooms"],
        },
    }
]

## Tool Implementation

This is where you implement the actual tool logic. In production, this would hit a real database or MLS API. Here we use sample data for demonstration.

In [ ]:
SAMPLE_SALES_DATA = {
    "Kailua": [
        {"address": "123 Kailua Rd", "price": 1250000, "sqft": 1400, "beds": 3, "baths": 2, "sold": "2026-01-15"},
        {"address": "456 Oneawa St", "price": 1180000, "sqft": 1350, "beds": 3, "baths": 2, "sold": "2026-02-01"},
        {"address": "789 Kuulei Rd", "price": 1320000, "sqft": 1500, "beds": 3, "baths": 2.5, "sold": "2026-01-28"},
        {"address": "321 Hamakua Dr", "price": 1410000, "sqft": 1600, "beds": 4, "baths": 3, "sold": "2026-02-10"},
        {"address": "654 Kalaheo Ave", "price": 1150000, "sqft": 1280, "beds": 3, "baths": 2, "sold": "2026-01-20"},
    ],
    "Manoa": [
        {"address": "100 Manoa Rd", "price": 980000, "sqft": 1100, "beds": 2, "baths": 1, "sold": "2026-01-10"},
        {"address": "200 Oahu Ave", "price": 1050000, "sqft": 1250, "beds": 3, "baths": 2, "sold": "2026-02-05"},
        {"address": "300 Lowrey Ave", "price": 1120000, "sqft": 1300, "beds": 3, "baths": 2, "sold": "2026-01-22"},
    ],
}


def lookup_comparable_sales(neighborhood: str, bedrooms: int, max_results: int = 5) -> str:
    """
    Simulates looking up comparable sales. In production, this would query
    an MLS database or API. Returns JSON string of results.
    """
    sales = SAMPLE_SALES_DATA.get(neighborhood, [])
    # Filter by bedroom count
    filtered = [s for s in sales if s["beds"] == bedrooms]
    # Limit results
    filtered = filtered[:max_results]

    if not filtered:
        return json.dumps({"error": f"No comparable sales found for {bedrooms}BR in {neighborhood}"})

    return json.dumps({"neighborhood": neighborhood, "comparable_sales": filtered, "count": len(filtered)})

## System Prompt

This is the agent's "job description." Notice how it:
1. Defines the role clearly
2. Sets specific rules for behavior
3. Includes few-shot examples showing the expected analysis pattern
4. Emphasizes evidence-based reasoning (KEY LESSON!)

In [ ]:
SYSTEM_PROMPT = """You are a residential real estate pricing analyst specializing in Hawaii properties.

Your job is to help agents determine competitive listing prices by analyzing comparable sales data.

## Rules
- ALWAYS use the lookup_comparable_sales tool to get real data before making recommendations
- NEVER guess at prices without evidence -- your recommendations must be grounded in comparable sales
- Calculate price per square foot from comparables to support your analysis
- Consider property features (beds, baths, sqft) when adjusting from comparables
- Provide a recommended price RANGE, not a single number
- Explain your reasoning clearly, citing specific comparables

## Output Format
Structure your response as:
1. **Comparable Sales Summary** -- list the comps you found
2. **Price Per Square Foot Analysis** -- calculate and compare $/sqft
3. **Recommended Listing Price Range** -- your recommendation with rationale
4. **Key Considerations** -- factors that could push the price higher or lower

## Few-Shot Examples

### Example Analysis 1:
User asked about a 3BR/2BA, 1,300 sqft home in Kailua.
Comparables found:
- 123 Main St: $1,200,000 (1,350 sqft = $889/sqft)
- 456 Oak Ave: $1,150,000 (1,280 sqft = $898/sqft)
- 789 Palm Dr: $1,280,000 (1,400 sqft = $914/sqft)

Average $/sqft: $900
Subject property: 1,300 sqft x $900/sqft = $1,170,000

Recommended range: $1,150,000 - $1,200,000
Reasoning: The subject property is mid-range in size among the comps. No premium features noted, so pricing at the average $/sqft is appropriate.

### Example Analysis 2:
User asked about a 4BR/3BA, 1,800 sqft home in Hawaii Kai with ocean view.
Comparables found:
- 100 Marina Dr: $1,500,000 (1,750 sqft = $857/sqft)
- 200 Bay St: $1,650,000 (1,900 sqft = $868/sqft, ocean view)

Average $/sqft: $863 (but ocean view comps trend 5-10% higher)
Subject property: 1,800 sqft x $910/sqft (adjusted for view) = $1,638,000

Recommended range: $1,600,000 - $1,680,000
Reasoning: Ocean view commands a premium. Adjusted $/sqft upward based on the view comp."""

## Process Tool Calls

When the model wants to use a tool, it returns a `tool_use` content block instead of (or alongside) text. We need to:
1. Detect the tool call
2. Execute the tool with the provided arguments
3. Send the result back to the model so it can continue reasoning

In [ ]:
def process_tool_call(tool_name: str, tool_input: dict) -> str:
    """Execute a tool and return its result as a string."""
    if tool_name == "lookup_comparable_sales":
        return lookup_comparable_sales(
            neighborhood=tool_input["neighborhood"],
            bedrooms=tool_input["bedrooms"],
            max_results=tool_input.get("max_results", 5),
        )
    else:
        return json.dumps({"error": f"Unknown tool: {tool_name}"})

## The Agent Loop

This is the core of any agent. It works like this:
1. Send the user's message to the model
2. Check if the model wants to use a tool
3. If yes: execute the tool, send the result back, go to step 2
4. If no: return the model's text response (we're done)

The loop continues until the model produces a final text response without requesting any more tool calls.

In [ ]:
def run_agent(user_message: str) -> str:
    """
    Run the property analysis agent with the given user message.
    Returns the agent's final text response.
    """
    print(f"\n{'='*60}")
    print(f"User: {user_message}")
    print(f"{'='*60}\n")

    # Build the initial messages list
    messages = [{"role": "user", "content": user_message}]

    # Agent loop -- keep going until we get a final response
    while True:
        # Call the API
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            tools=tools,
            messages=messages,
        )

        # Check the stop reason to decide what to do next
        if response.stop_reason == "tool_use":
            # The model wants to call a tool.
            # First, add the assistant's response to the conversation history.
            messages.append({"role": "assistant", "content": response.content})

            # Find and execute each tool call
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  [Tool Call] {block.name}({json.dumps(block.input)})")

                    # Execute the tool
                    result = process_tool_call(block.name, block.input)
                    print(f"  [Tool Result] {result[:200]}...")

                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result,
                        }
                    )

            # Send the tool results back to the model
            messages.append({"role": "user", "content": tool_results})

        elif response.stop_reason == "end_turn":
            # The model is done -- extract and return the final text
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text += block.text
            return final_text

        else:
            # Unexpected stop reason
            return f"Agent stopped unexpectedly: {response.stop_reason}"

## Run the Agent

Notice how the user provides SPECIFIC details about their property. This is evidence-based prompting in action -- we're giving the agent concrete data to reason about, not asking it to guess.

In [ ]:
# Evidence-based prompt: we provide specific property details
user_query = (
    "I have a 3-bedroom, 2-bathroom home in Kailua that is 1,375 square feet. "
    "It was recently renovated with a new kitchen and has a partial ocean view. "
    "What should I list it for?"
)

result = run_agent(user_query)

print(f"\n{'='*60}")
print("Agent's Analysis:")
print(f"{'='*60}")
print(result)